# 01 - Data Profiling & Quality Checks
Dataset: `data/raw.csv`. Result at `docs/data_quality_findings.md`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

CSV_PATH = Path('../data/raw.csv')
if not CSV_PATH.exists():
    CSV_PATH = Path('data/raw.csv')

df = pd.read_csv(CSV_PATH, parse_dates=['date'])
print('Loaded:', df.shape)
df.head()

## 1. Shape, dtypes, memory

In [ ]:
print('Rows, Cols:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nDate range:', df['date'].min().date(), '->', df['date'].max().date())
print('Distinct days:', df['date'].nunique())
df.info(memory_usage='deep')

## 2. Missing values

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_tbl = pd.DataFrame({'n_missing': missing, 'pct': missing_pct})
miss_tbl

## 3. Duplicate business key

In [ ]:
key = ['date', 'sku', 'channel', 'region', 'pack_type']
dups = df.duplicated(subset=key).sum()
print('Duplicate key rows:', dups)
full_dups = df.duplicated().sum()
print('Fully-duplicated rows:', full_dups)

## 4. Numeric ranges & invalid values

In [ ]:
print(df[['price_unit','delivery_days','stock_available','delivered_qty','units_sold']].describe())

neg = {
    'price_unit<0': (df['price_unit'] < 0).sum(),
    'units_sold<0': (df['units_sold'] < 0).sum(),
    'delivered_qty<0': (df['delivered_qty'] < 0).sum(),
    'stock_available<0': (df['stock_available'] < 0).sum(),
    'delivery_days<0': (df['delivery_days'] < 0).sum(),
    'promo not 0/1': (~df['promotion_flag'].isin([0,1])).sum(),
}
print('\nInvalid value counts:')
for k,v in neg.items():
    print(f'  {k:20s}: {v}')

## 5. Categorical vocabulary

In [ ]:
for col in ['category','segment','brand','channel','region','pack_type']:
    vals = df[col].nunique()
    print(f'{col}: {vals} distinct')
print()
print('categories:', sorted(df['category'].unique()))
print('channels:', sorted(df['channel'].unique()))
print('regions:', sorted(df['region'].unique()))
print('pack_types:', sorted(df['pack_type'].unique()))
print('n SKUs:', df['sku'].nunique())
print('n brands:', df['brand'].nunique())

## 6. Product hierarchy consistency

In [ ]:
h = df.groupby('sku').agg(n_brand=('brand','nunique'),
                          n_segment=('segment','nunique'),
                          n_category=('category','nunique'))
bad_h = h[(h['n_brand']>1)|(h['n_segment']>1)|(h['n_category']>1)]
print('SKUs with inconsistent hierarchy:', len(bad_h))
bad_h.head()

## 7. Stock-out/ zero-sales analysis

In [ ]:
zero_sales = (df['units_sold'] == 0).sum()
zero_no_stock = ((df['units_sold']==0) & (df['stock_available']==0)).sum()
zero_with_stock = ((df['units_sold']==0) & (df['stock_available']>0)).sum()
sold_gt_stock = (df['units_sold'] > df['stock_available']).sum()
print(f'Zero-sales rows: {zero_sales:,} ({zero_sales/len(df)*100:.1f}%)')
print(f'  ...with no stock : {zero_no_stock:,}')
print(f'  ...with stock > 0 : {zero_with_stock:,}')
print(f'units_sold > stock : {sold_gt_stock:,}  (investigate logic)')

## 8. Product introductions, first appearance year per SKU

In [ ]:
first_year = df.groupby('sku')['date'].min().dt.year
print(first_year.value_counts().sort_index())
new_2024 = first_year[first_year == 2024]
print(f'\nSKUs first seen in 2024 (cold-start): {len(new_2024)}')
print(list(new_2024.index[:20]))

## 9. Price drift across years

In [ ]:
df['year'] = df['date'].dt.year
price_yr = df.groupby('year')['price_unit'].agg(['mean','min','max']).round(2)
print(price_yr)

## 10. Write Data Quality Findings summary to docs

In [ ]:
out = Path('../docs/data_quality_findings.md')
out.parent.mkdir(parents=True, exist_ok=True)
lines = []
lines.append('# Data Quality Findings\n')
lines.append(f'- Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
lines.append(f'- Date range: {df.date.min().date()} -> {df.date.max().date()} ({df.date.nunique()} distinct days)')
lines.append(f'- Missing values: {int(df.isna().sum().sum())} total')
lines.append(f'- Duplicate key rows: {int(df.duplicated(subset=key).sum())}')
lines.append(f'- Negative/invalid numeric rows: {sum(neg.values())}')
lines.append(f'- SKUs: {df.sku.nunique()} | Brands: {df.brand.nunique()} | Categories: {df.category.nunique()}')
lines.append(f'- SKUs inconsistent hierarchy: {len(bad_h)}')
lines.append(f'- Zero-sales rows: {zero_sales:,} ({zero_sales/len(df)*100:.1f}%); of which stock>0: {zero_with_stock:,}')
lines.append(f'- units_sold > stock_available: {sold_gt_stock:,}')
lines.append(f'- SKUs first seen in 2024: {len(new_2024)}')
lines.append('\n## Price by year\n')
lines.append(price_yr.to_markdown())
out.write_text('\n'.join(lines), encoding='utf-8')
print('Wrote', out.resolve())
print('\n'.join(lines))